# NLP Assignment 3 - Transformers + RAG
Waleed Saeed (21i-1672)

In [2]:
import json
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter

# 1. Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# 2. Load the pre-sampled data
with open('sample_data/sampled_dataset.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data)} reviews from sampled dataset.")

# 3. Feature Extraction & Sentiment Mapping
processed_data = []
for item in raw_data:
    text = item['text']
    rating = item['rating']

    # Derived Feature: Word Count
    word_count = len(text.split())

    # Sentiment Mapping
    if rating <= 2.0:
        sentiment = 0 # Negative
    elif rating == 3.0:
        sentiment = 1 # Neutral
    else:
        sentiment = 2 # Positive

    processed_data.append({
        'text': text,
        'sentiment': sentiment,
        'length': word_count
    })

# 4. Train/Val/Test Split (70%, 15%, 15%)
train_data, temp_data = train_test_split(processed_data, test_size=0.30, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.50, random_state=42)
print(f"Splits -> Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

# 5. Vocabulary Construction (TRAIN ONLY)
vocab = {'<PAD>': 0, '<UNK>': 1, '<CLS>': 2}
word_counts = Counter()

for item in train_data:
    tokens = item['text'].lower().split()
    word_counts.update(tokens)

# Keep top 15,000 words to keep embedding layer manageable
for word, count in word_counts.most_common(15000):
    vocab[word] = len(vocab)
print(f"Vocabulary size: {len(vocab)} tokens")

# 6. PyTorch Dataset Class
MAX_SEQ_LEN = 128

class AmazonReviewDataset(Dataset):
    def __init__(self, data, vocab, max_len):
        self.data = data
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item['text'].lower().split()

        # Convert to indices
        indices = [self.vocab.get(w, self.vocab['<UNK>']) for w in tokens]

        # Add CLS token at the start
        indices = [self.vocab['<CLS>']] + indices

        # Pad or Truncate
        if len(indices) > self.max_len:
            indices = indices[:self.max_len]
            mask = [1] * self.max_len
        else:
            pad_len = self.max_len - len(indices)
            mask = [1] * len(indices) + [0] * pad_len
            indices = indices + [self.vocab['<PAD>']] * pad_len

        return {
            'input_ids': torch.tensor(indices, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'sentiment': torch.tensor(item['sentiment'], dtype=torch.long),
            'derived_feature': torch.tensor(item['length'], dtype=torch.float), # Float for MSE loss
            'raw_text': item['text'] # Needed later for Part C (Decoder)
        }

# 7. Create DataLoaders
batch_size = 64 # Good balance for speed on a GPU

train_dataset = AmazonReviewDataset(train_data, vocab, MAX_SEQ_LEN)
val_dataset = AmazonReviewDataset(val_data, vocab, MAX_SEQ_LEN)
test_dataset = AmazonReviewDataset(test_data, vocab, MAX_SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print("DataLoaders ready! Input shape:", next(iter(train_loader))['input_ids'].shape)

Loaded 30000 reviews from sampled dataset.
Splits -> Train: 21000 | Val: 4500 | Test: 4500
Vocabulary size: 15003 tokens
DataLoaders ready! Input shape: torch.Size([64, 128])
